# Data Wrangling Assignment
## Building a Modular Data Sanitization & Exploration Engine

This notebook implements two classes:
- `DataInspector`: End-to-end data ingestion, cleaning, feature engineering, and visualization.
- `PlottingMethods`: Standalone granular chart generation.

**Full demo uses the Titanic dataset**: Upload → Impute → Normalize → Visualize Associations.

## 1. Install Dependencies

In [ ]:
!pip install -q plotly scipy scikit-learn pandas numpy

## 2. Core Implementation

In [ ]:
import io
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from scipy.stats import pointbiserialr, f_oneway
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from IPython.display import display, HTML

warnings.filterwarnings('ignore')

# ---------------------------------------------------------------------------
# NULL / garbage strings to treat as NaN on load
# ---------------------------------------------------------------------------
_NULL_STRINGS = ['?', 'n/a', 'na', 'null', 'none', 'nan', '', ' ', 'N/A',
                 'NULL', 'None', 'NaN', 'missing', 'MISSING', '-', '--']


# ===========================================================================
# PlottingMethods – standalone granular chart factory
# ===========================================================================
class PlottingMethods:
    """Standalone chart generation class returning Plotly HTML for embedding.

    Every public `plot_*` method accepts either a ``data`` keyword argument
    (a pandas DataFrame or JSON-serialisable structure) or falls back to
    ``self.df`` if it is set.  They all return a dict with keys:
        - ``status``  : 'success' or 'error'
        - ``html``    : full-page HTML string (on success)
        - ``message`` : error description (on error)
    """

    def __init__(self):
        self.df: pd.DataFrame | None = None

    # -----------------------------------------------------------------------
    # Internal helpers
    # -----------------------------------------------------------------------
    def _resolve_df(self, data) -> pd.DataFrame | None:
        """Return a DataFrame from *data* or self.df, or None."""
        if data is not None:
            if isinstance(data, pd.DataFrame):
                return data
            try:
                return pd.DataFrame(data)
            except Exception:
                return None
        return self.df

    @staticmethod
    def _wrap(fig) -> dict:
        """Wrap a Plotly figure in the standard return dict."""
        return {'status': 'success', 'html': fig.to_html(full_html=True, include_plotlyjs='cdn')}

    @staticmethod
    def _err(msg: str) -> dict:
        return {'status': 'error', 'message': msg}

    # -----------------------------------------------------------------------
    # Display helper
    # -----------------------------------------------------------------------
    def display_image(self, result: dict):
        """Render the HTML returned by any plot_* method inside Colab/Jupyter."""
        if result.get('status') == 'success':
            display(HTML(result['html']))
        else:
            print(f"[PlottingMethods] Error: {result.get('message', 'Unknown error')}")

    # -----------------------------------------------------------------------
    # Methods info
    # -----------------------------------------------------------------------
    def get_methods_info(self) -> dict:
        """Return a summary of all public plotting methods and their parameters."""
        info = [
            {'Method': 'plot_bar_chart',      'Required': 'x, y',              'Optional': 'color, barmode, title, data'},
            {'Method': 'plot_pie_chart',      'Required': 'names, values',      'Optional': 'hole, title, data'},
            {'Method': 'plot_histogram',      'Required': 'x',                  'Optional': 'bins, color, title, data'},
            {'Method': 'plot_scatter',        'Required': 'x, y',              'Optional': 'color, trendline, title, data'},
            {'Method': 'plot_box',            'Required': 'x, y',              'Optional': 'color, title, data'},
            {'Method': 'plot_heat_map',       'Required': 'values,index,columns','Optional': 'aggregade_method, title, data'},
            {'Method': 'plot_sankey_diagram', 'Required': 'source_column, target_column, values', 'Optional': 'title, data'},
            {'Method': 'plot_simple_sunburst_graph', 'Required': 'path, values', 'Optional': 'title, data'},
        ]
        return {'response': info}

    # -----------------------------------------------------------------------
    # Individual chart methods
    # -----------------------------------------------------------------------
    def plot_bar_chart(self, x: str, y: str, color: str = None,
                       barmode: str = 'group', title: str = None,
                       data=None) -> dict:
        """Grouped or stacked bar chart.

        Parameters
        ----------
        x       : column name for the x-axis categories.
        y       : column name for bar heights.
        color   : optional column for colour grouping.
        barmode : 'group' (default) or 'stack'.
        title   : chart title.
        data    : DataFrame or JSON-serialisable object (overrides self.df).
        """
        df = self._resolve_df(data)
        if df is None or df.empty:
            return self._err('No data provided.')
        if x not in df.columns or y not in df.columns:
            return self._err(f'Columns {x!r} or {y!r} not found.')
        fig = px.bar(df, x=x, y=y, color=color, barmode=barmode,
                     title=title or f'{y} by {x}',
                     template='plotly_white')
        fig.update_layout(height=450)
        return self._wrap(fig)

    def plot_pie_chart(self, names: str, values: str, hole: float = 0.0,
                       title: str = None, data=None) -> dict:
        """Responsive pie / donut chart.

        Parameters
        ----------
        names  : column supplying slice labels.
        values : column supplying slice sizes.
        hole   : donut hole ratio 0–1 (0 = solid pie).
        title  : chart title.
        data   : DataFrame or JSON-serialisable object.
        """
        df = self._resolve_df(data)
        if df is None or df.empty:
            return self._err('No data provided.')
        if names not in df.columns or values not in df.columns:
            return self._err(f'Columns {names!r} or {values!r} not found.')
        agg = df.groupby(names)[values].sum().reset_index()
        fig = px.pie(agg, names=names, values=values, hole=hole,
                     title=title or f'{values} by {names}',
                     template='plotly_white')
        fig.update_layout(height=450)
        return self._wrap(fig)

    def plot_histogram(self, x: str, bins=None, color: str = None,
                       title: str = None, data=None) -> dict:
        """Distribution histogram with optional custom bin boundaries.

        Parameters
        ----------
        x     : column to plot.
        bins  : list of bin-edge values, or integer number of bins.
        color : optional grouping column.
        title : chart title.
        data  : DataFrame or JSON-serialisable object.
        """
        df = self._resolve_df(data)
        if df is None or df.empty:
            return self._err('No data provided.')
        if x not in df.columns:
            return self._err(f'Column {x!r} not found.')
        xbins = {}
        if isinstance(bins, (list, np.ndarray)) and len(bins) >= 2:
            xbins = dict(start=bins[0], end=bins[-1], size=(bins[-1]-bins[0])/max(len(bins)-1, 1))
        elif isinstance(bins, int):
            rng = df[x].dropna()
            xbins = dict(start=rng.min(), end=rng.max(), size=(rng.max()-rng.min())/bins)
        fig = px.histogram(df, x=x, color=color, nbins=bins if isinstance(bins, int) else None,
                           title=title or f'Distribution of {x}',
                           template='plotly_white')
        if xbins:
            fig.update_traces(xbins=xbins)
        fig.update_layout(height=420)
        return self._wrap(fig)

    def plot_scatter(self, x: str, y: str, color: str = None,
                     trendline: str = None, title: str = None,
                     data=None) -> dict:
        """Scatter plot with optional trendline.

        Parameters
        ----------
        x, y       : numeric column names.
        color      : optional grouping column.
        trendline  : e.g. 'ols' for OLS regression line.
        title      : chart title.
        data       : DataFrame or JSON-serialisable object.
        """
        df = self._resolve_df(data)
        if df is None or df.empty:
            return self._err('No data provided.')
        fig = px.scatter(df, x=x, y=y, color=color, trendline=trendline,
                         title=title or f'{y} vs {x}',
                         template='plotly_white')
        fig.update_layout(height=450)
        return self._wrap(fig)

    def plot_box(self, x: str, y: str, color: str = None,
                 title: str = None, data=None) -> dict:
        """Box plot (with all data points overlaid).

        Parameters
        ----------
        x     : categorical column.
        y     : numeric column.
        color : optional grouping column.
        title : chart title.
        data  : DataFrame or JSON-serialisable object.
        """
        df = self._resolve_df(data)
        if df is None or df.empty:
            return self._err('No data provided.')
        fig = px.box(df, x=x, y=y, color=color, points='all',
                     title=title or f'{y} by {x}',
                     template='plotly_white')
        fig.update_layout(height=480)
        return self._wrap(fig)

    def plot_heat_map(self, values: str, index: str, columns: str,
                      aggregade_method: str = 'mean',
                      title: str = None, data=None) -> dict:
        """Pivot-table heatmap.

        Parameters
        ----------
        values           : numeric column to aggregate.
        index            : row grouping column.
        columns          : column grouping column.
        aggregade_method : pandas aggfunc string ('mean', 'sum', 'count', …).
        title            : chart title.
        data             : DataFrame or JSON-serialisable object.
        """
        df = self._resolve_df(data)
        if df is None or df.empty:
            return self._err('No data provided.')
        pivot = df.pivot_table(values=values, index=index, columns=columns,
                               aggfunc=aggregade_method)
        fig = px.imshow(pivot, text_auto='.2f', aspect='auto',
                        title=title or f'{values} ({aggregade_method}) by {index} × {columns}',
                        color_continuous_scale='RdBu_r',
                        template='plotly_white')
        fig.update_layout(height=500)
        return self._wrap(fig)

    def plot_sankey_diagram(self, source_column: str, target_column: str,
                            values: str, title: str = None,
                            data=None) -> dict:
        """Sankey flow diagram.

        Parameters
        ----------
        source_column : column with flow source labels.
        target_column : column with flow target labels.
        values        : numeric column for flow magnitude.
        title         : chart title.
        data          : DataFrame or JSON-serialisable object.
        """
        df = self._resolve_df(data)
        if df is None or df.empty:
            return self._err('No data provided.')
        agg = df.groupby([source_column, target_column])[values].sum().reset_index()
        all_nodes = pd.unique(agg[[source_column, target_column]].values.ravel())
        node_idx = {n: i for i, n in enumerate(all_nodes)}
        fig = go.Figure(go.Sankey(
            node=dict(label=list(all_nodes), pad=15, thickness=20),
            link=dict(
                source=agg[source_column].map(node_idx).tolist(),
                target=agg[target_column].map(node_idx).tolist(),
                value=agg[values].tolist()
            )
        ))
        fig.update_layout(title_text=title or f'{source_column} → {target_column}',
                          height=500, template='plotly_white')
        return self._wrap(fig)

    def plot_simple_sunburst_graph(self, path: list, values: str,
                                   title: str = None, data=None) -> dict:
        """Sunburst (hierarchical pie) chart.

        Parameters
        ----------
        path   : list of column names forming the hierarchy (outer → inner).
        values : numeric column for sector sizes.
        title  : chart title.
        data   : DataFrame or JSON-serialisable object.
        """
        df = self._resolve_df(data)
        if df is None or df.empty:
            return self._err('No data provided.')
        missing_cols = [c for c in path if c not in df.columns]
        if missing_cols:
            return self._err(f'Columns not found: {missing_cols}')
        fig = px.sunburst(df, path=path, values=values,
                          title=title or ' / '.join(path),
                          template='plotly_white')
        fig.update_layout(height=550)
        return self._wrap(fig)


# ===========================================================================
# DataInspector – end-to-end data pipeline
# ===========================================================================
class DataInspector(PlottingMethods):
    """All-in-one data cleaning, exploration, and visualisation toolkit.

    Inherits `PlottingMethods` so all chart helpers are directly accessible
    via ``inspector.plot_bar_chart(...)`` etc.

    Attributes
    ----------
    df : pd.DataFrame
        The active working dataset.  Assign directly or use `upload_data()`.
    """

    def __init__(self):
        super().__init__()

    # -----------------------------------------------------------------------
    # 1. Data Ingestion
    # -----------------------------------------------------------------------
    def upload_data(self):
        """Interactively upload a CSV file in Google Colab.

        Automatically replaces common garbage strings with NaN and
        attempts numeric coercion for all columns.
        """
        try:
            from google.colab import files
            uploaded = files.upload()
        except ImportError:
            print('[upload_data] Not running in Colab – assign a DataFrame to inspector.df directly.')
            return
        for fname, content in uploaded.items():
            self.df = pd.read_csv(io.BytesIO(content), na_values=_NULL_STRINGS,
                                  keep_default_na=True)
            self._auto_convert_types()
            print(f'[upload_data] Loaded {fname!r} → {self.df.shape[0]} rows × {self.df.shape[1]} cols')
            break  # only first file

    def _auto_convert_types(self):
        """Attempt numeric coercion; keep original if conversion yields all NaN."""
        if self.df is None:
            return
        for col in self.df.select_dtypes(include='object').columns:
            converted = pd.to_numeric(self.df[col], errors='coerce')
            if converted.notna().sum() > 0:
                self.df[col] = converted

    # -----------------------------------------------------------------------
    # 2. Structural Analysis & Cleaning
    # -----------------------------------------------------------------------
    def get_summary(self):
        """Print shape, column type breakdown, statistical summary, and first 20 rows."""
        if self.df is None or self.df.empty:
            print('[get_summary] No data loaded.'); return
        num_cols = self.df.select_dtypes(include='number').columns.tolist()
        cat_cols = self.df.select_dtypes(exclude='number').columns.tolist()
        print('=' * 60)
        print(f'Shape          : {self.df.shape[0]} rows × {self.df.shape[1]} columns')
        print(f'Numeric cols   : {len(num_cols)}  → {num_cols}')
        print(f'Categorical cols: {len(cat_cols)}  → {cat_cols}')
        print('=' * 60)
        print('\n--- Numeric Statistical Summary ---')
        display(self.df[num_cols].describe().T)
        print('\n--- First 20 Rows ---')
        display(self.df.head(20))

    def column_details(self):
        """Display per-column dtype, null count, null %, and unique value count."""
        if self.df is None or self.df.empty:
            print('[column_details] No data loaded.'); return
        info = pd.DataFrame({
            'dtype':       self.df.dtypes,
            'null_count':  self.df.isna().sum(),
            'null_pct':    (self.df.isna().mean() * 100).round(2),
            'unique':      self.df.nunique()
        })
        display(info)

    def get_categorical_summary(self):
        """Print value counts and top-5 values for each categorical column."""
        if self.df is None or self.df.empty:
            print('[get_categorical_summary] No data loaded.'); return
        cat_cols = self.df.select_dtypes(exclude='number').columns
        if len(cat_cols) == 0:
            print('No categorical columns found.'); return
        for col in cat_cols:
            print(f'\n--- {col} ---')
            display(self.df[col].value_counts().head(10).to_frame('count'))

    def show_missing_data(self):
        """Display a bar chart of missing value counts per column."""
        if self.df is None or self.df.empty:
            print('[show_missing_data] No data loaded.'); return
        missing = self.df.isna().sum()
        missing = missing[missing > 0].sort_values(ascending=False)
        if missing.empty:
            print('No missing values found!'); return
        fig = px.bar(x=missing.index, y=missing.values,
                     labels={'x': 'Column', 'y': 'Missing Count'},
                     title='Missing Values per Column',
                     template='plotly_white', color=missing.values,
                     color_continuous_scale='reds')
        fig.show()

    def handle_missing_values(self, strategy: str = 'mean', constant=0,
                               columns: list = None):
        """Impute missing values in-place.

        Parameters
        ----------
        strategy : 'mean', 'median', 'mode', or 'constant'.
        constant : value used when strategy='constant'.
        columns  : list of column names to impute (all if None).
        """
        if self.df is None or self.df.empty:
            print('[handle_missing_values] No data loaded.'); return
        cols = columns if columns else self.df.columns.tolist()
        for col in cols:
            if self.df[col].isna().sum() == 0:
                continue
            if strategy == 'mean':
                if pd.api.types.is_numeric_dtype(self.df[col]):
                    self.df[col].fillna(self.df[col].mean(), inplace=True)
            elif strategy == 'median':
                if pd.api.types.is_numeric_dtype(self.df[col]):
                    self.df[col].fillna(self.df[col].median(), inplace=True)
            elif strategy == 'mode':
                mode_val = self.df[col].mode()
                if not mode_val.empty:
                    self.df[col].fillna(mode_val[0], inplace=True)
            elif strategy == 'constant':
                self.df[col].fillna(constant, inplace=True)
            else:
                print(f'[handle_missing_values] Unknown strategy: {strategy!r}')
                return
        remaining = self.df[cols].isna().sum().sum()
        print(f'[handle_missing_values] Strategy={strategy!r} applied. '
              f'Remaining nulls in scope: {remaining}')

    def remove_duplicates(self):
        """Remove exact duplicate rows and report how many were dropped."""
        if self.df is None or self.df.empty:
            print('[remove_duplicates] No data loaded.'); return
        before = len(self.df)
        self.df.drop_duplicates(inplace=True)
        self.df.reset_index(drop=True, inplace=True)
        print(f'[remove_duplicates] Removed {before - len(self.df)} duplicate rows. '
              f'Remaining: {len(self.df)}')

    def handle_outliers(self, columns: list = None, find_and_delete: bool = False,
                        iqr_multiplier: float = 1.5):
        """Detect and optionally remove outliers using the IQR method.

        Parameters
        ----------
        columns         : numeric columns to inspect (all numeric if None).
        find_and_delete : True → drop outlier rows; False → report only.
        iqr_multiplier  : fence multiplier (default 1.5 → standard IQR rule).
        """
        if self.df is None or self.df.empty:
            print('[handle_outliers] No data loaded.'); return
        num_cols = self.df.select_dtypes(include='number').columns.tolist()
        cols = columns if columns else num_cols
        cols = [c for c in cols if c in num_cols]
        if not cols:
            print('[handle_outliers] No numeric columns to inspect.'); return
        mask = pd.Series([False] * len(self.df), index=self.df.index)
        for col in cols:
            q1, q3 = self.df[col].quantile([0.25, 0.75])
            iqr = q3 - q1
            lo, hi = q1 - iqr_multiplier * iqr, q3 + iqr_multiplier * iqr
            col_mask = (self.df[col] < lo) | (self.df[col] > hi)
            print(f'  {col}: {col_mask.sum()} outliers  (fence [{lo:.2f}, {hi:.2f}])')
            mask |= col_mask
        total = mask.sum()
        if find_and_delete:
            self.df = self.df[~mask].reset_index(drop=True)
            print(f'[handle_outliers] Deleted {total} outlier rows. Remaining: {len(self.df)}')
        else:
            print(f'[handle_outliers] Found {total} outlier rows (not deleted – '
                  'pass find_and_delete=True to remove).')

    def delete_rows(self, indices: str = None):
        """Interactively or programmatically delete rows by index.

        Parameters
        ----------
        indices : comma-separated row indices to delete (prompt if None).
        """
        if self.df is None or self.df.empty:
            print('[delete_rows] No data loaded.'); return
        if indices is None:
            indices = input('Enter comma-separated row indices to delete: ')
        try:
            idx_list = [int(i.strip()) for i in indices.split(',') if i.strip()]
        except ValueError:
            print('[delete_rows] Invalid input – please provide integers.'); return
        valid = [i for i in idx_list if i in self.df.index]
        self.df.drop(index=valid, inplace=True)
        self.df.reset_index(drop=True, inplace=True)
        print(f'[delete_rows] Deleted rows {valid}. Remaining: {len(self.df)}')

    def delete_columns(self, columns: str = None):
        """Interactively or programmatically delete columns by name.

        Parameters
        ----------
        columns : comma-separated column names to drop (prompt if None).
        """
        if self.df is None or self.df.empty:
            print('[delete_columns] No data loaded.'); return
        if columns is None:
            print('Existing columns:', list(self.df.columns))
            columns = input('Enter comma-separated column names to delete: ')
        col_list = [c.strip() for c in columns.split(',') if c.strip()]
        valid = [c for c in col_list if c in self.df.columns]
        invalid = [c for c in col_list if c not in self.df.columns]
        if invalid:
            print(f'[delete_columns] Columns not found (skipped): {invalid}')
        self.df.drop(columns=valid, inplace=True)
        print(f'[delete_columns] Dropped: {valid}. Remaining columns: {list(self.df.columns)}')

    # -----------------------------------------------------------------------
    # 3. Feature Engineering Preparation
    # -----------------------------------------------------------------------
    def extract_normalized_numeric_data(self, method: str = 'minmax',
                                        columns: list = None) -> pd.DataFrame:
        """Return a scaled copy of numeric columns.

        Parameters
        ----------
        method  : 'minmax', 'standard' (Z-score), or 'robust' (IQR-based).
        columns : columns to scale (all numeric if None).

        Returns
        -------
        pd.DataFrame with scaled numeric columns.
        """
        if self.df is None or self.df.empty:
            print('[extract_normalized_numeric_data] No data.'); return pd.DataFrame()
        num_cols = columns if columns else self.df.select_dtypes(include='number').columns.tolist()
        num_df = self.df[num_cols].copy()
        scalers = {'minmax': MinMaxScaler(), 'standard': StandardScaler(),
                   'robust': RobustScaler()}
        if method not in scalers:
            print(f'[extract_normalized_numeric_data] Unknown method {method!r}. '
                  f'Choose from {list(scalers)}.'); return pd.DataFrame()
        scaler = scalers[method]
        scaled = scaler.fit_transform(num_df.fillna(num_df.median()))
        result = pd.DataFrame(scaled, columns=[f'{c}_scaled' for c in num_df.columns])
        print(f'[extract_normalized_numeric_data] {method} scaling applied to {len(num_cols)} columns.')
        return result

    def extract_normalized_categorical_data(self, method: str = 'onehot',
                                            columns: list = None) -> pd.DataFrame:
        """Return an encoded copy of categorical columns.

        Parameters
        ----------
        method  : 'onehot', 'ordinal', or 'uniform' (scaled 0–1 label).
        columns : columns to encode (all categorical if None).

        Returns
        -------
        pd.DataFrame with encoded categorical columns.
        """
        if self.df is None or self.df.empty:
            print('[extract_normalized_categorical_data] No data.'); return pd.DataFrame()
        cat_cols = columns if columns else self.df.select_dtypes(exclude='number').columns.tolist()
        if not cat_cols:
            print('[extract_normalized_categorical_data] No categorical columns.'); return pd.DataFrame()
        cat_df = self.df[cat_cols].copy().fillna('MISSING')
        if method == 'onehot':
            result = pd.get_dummies(cat_df, prefix=cat_cols)
        elif method == 'ordinal':
            result = cat_df.apply(lambda s: s.astype('category').cat.codes)
            result.columns = [f'{c}_ord' for c in result.columns]
        elif method == 'uniform':
            result = cat_df.apply(lambda s: s.astype('category').cat.codes)
            result = result.apply(lambda s: s / max(s.max(), 1))
            result.columns = [f'{c}_uni' for c in result.columns]
        else:
            print(f'[extract_normalized_categorical_data] Unknown method {method!r}.')
            return pd.DataFrame()
        print(f'[extract_normalized_categorical_data] {method} encoding applied to {len(cat_cols)} columns.')
        return result

    def create_normalized_data_df(self, num_method: str = 'minmax',
                                   cat_method: str = 'onehot') -> pd.DataFrame:
        """Merge scaled numeric and encoded categorical data into one DataFrame.

        Parameters
        ----------
        num_method : scaling method for numeric columns.
        cat_method : encoding method for categorical columns.

        Returns
        -------
        pd.DataFrame ready for ML training.
        """
        num_df = self.extract_normalized_numeric_data(method=num_method)
        cat_df = self.extract_normalized_categorical_data(method=cat_method)
        parts = [p for p in [num_df, cat_df] if not p.empty]
        if not parts:
            return pd.DataFrame()
        merged = pd.concat(parts, axis=1).reset_index(drop=True)
        print(f'[create_normalized_data_df] Final shape: {merged.shape}')
        return merged

    # -----------------------------------------------------------------------
    # 4. Advanced Interactive Visualisation
    # -----------------------------------------------------------------------
    def plot_numerical(self, column_names: list = None):
        """3-panel subplot (Violin, Scatter, Histogram) for each numeric column.

        Parameters
        ----------
        column_names : list of numeric columns to visualise (all if None).
        """
        if self.df is None or self.df.empty:
            print('[plot_numerical] No data loaded.'); return
        num_cols = self.df.select_dtypes(include='number').columns.tolist()
        cols = column_names if column_names else num_cols
        cols = [c for c in cols if c in num_cols]
        for col in cols:
            s = self.df[col].dropna()
            fig = make_subplots(rows=1, cols=3,
                                subplot_titles=[f'Violin – {col}',
                                                f'Scatter – {col}',
                                                f'Histogram – {col}'])
            fig.add_trace(go.Violin(x=s, name=col, box_visible=True,
                                    meanline_visible=True, orientation='h'), row=1, col=1)
            fig.add_trace(go.Scatter(x=s.index, y=s, mode='markers',
                                     marker=dict(size=4), name=col), row=1, col=2)
            fig.add_trace(go.Histogram(x=s, name=col), row=1, col=3)
            fig.update_layout(title_text=f'Distribution: {col}', height=350,
                               showlegend=False, template='plotly_white')
            fig.show()

    def plot_categorical(self, column_names: list = None):
        """Bar charts with raw counts and percentage labels for categorical columns.

        Parameters
        ----------
        column_names : list of categorical columns (all if None).
        """
        if self.df is None or self.df.empty:
            print('[plot_categorical] No data loaded.'); return
        cat_cols = self.df.select_dtypes(exclude='number').columns.tolist()
        cols = column_names if column_names else cat_cols
        cols = [c for c in cols if c in cat_cols]
        for col in cols:
            counts = self.df[col].value_counts().reset_index()
            counts.columns = [col, 'count']
            counts['pct'] = (counts['count'] / counts['count'].sum() * 100).round(1)
            counts['label'] = counts.apply(lambda r: f"{r['count']} ({r['pct']}%)", axis=1)
            fig = px.bar(counts, x=col, y='count', text='label',
                         title=f'Frequency: {col}', template='plotly_white')
            fig.update_traces(textposition='outside')
            fig.update_layout(height=420)
            fig.show()

    def plot_relationship(self, col1: str, col2: str):
        """Auto-select chart type based on column dtypes.

        - Num–Num   → scatter with OLS trendline
        - Cat–Num   → box plot with all data points
        - Cat–Cat   → grouped bar chart

        Parameters
        ----------
        col1, col2 : column names.
        """
        if self.df is None or self.df.empty:
            print('[plot_relationship] No data loaded.'); return
        for c in [col1, col2]:
            if c not in self.df.columns:
                print(f'[plot_relationship] Column {c!r} not found.'); return
        is_num1 = pd.api.types.is_numeric_dtype(self.df[col1])
        is_num2 = pd.api.types.is_numeric_dtype(self.df[col2])
        if is_num1 and is_num2:
            fig = px.scatter(self.df, x=col1, y=col2, trendline='ols',
                             title=f'{col2} vs {col1} (OLS)', template='plotly_white')
        elif not is_num1 and is_num2:
            fig = px.box(self.df, x=col1, y=col2, points='all',
                         title=f'{col2} by {col1}', template='plotly_white')
        elif is_num1 and not is_num2:
            fig = px.box(self.df, x=col2, y=col1, points='all',
                         title=f'{col1} by {col2}', template='plotly_white')
        else:
            ct = pd.crosstab(self.df[col1], self.df[col2]).reset_index().melt(id_vars=col1)
            fig = px.bar(ct, x=col1, y='value', color=col2, barmode='group',
                         title=f'{col1} vs {col2}', template='plotly_white')
        fig.update_layout(height=480)
        fig.show()

    # -----------------------------------------------------------------------
    # 5. Deep Statistical Insights
    # -----------------------------------------------------------------------
    def plot_numerical_correlation(self, method: str = 'pearson'):
        """Pearson (or Spearman/Kendall) correlation heatmap for numeric columns.

        Parameters
        ----------
        method : 'pearson', 'spearman', or 'kendall'.
        """
        if self.df is None or self.df.empty:
            print('[plot_numerical_correlation] No data.'); return
        num_df = self.df.select_dtypes(include='number')
        if num_df.shape[1] < 2:
            print('[plot_numerical_correlation] Need ≥ 2 numeric columns.'); return
        corr = num_df.corr(method=method)
        fig = px.imshow(corr, text_auto='.2f', aspect='auto',
                        color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
                        title=f'{method.capitalize()} Correlation Heatmap',
                        template='plotly_white')
        fig.update_layout(height=600)
        fig.show()

    def plot_categorical_correlation(self):
        """Cramér's V association heatmap for categorical columns."""
        if self.df is None or self.df.empty:
            print('[plot_categorical_correlation] No data.'); return
        cat_cols = self.df.select_dtypes(exclude='number').columns.tolist()
        if len(cat_cols) < 2:
            print('[plot_categorical_correlation] Need ≥ 2 categorical columns.'); return
        n = len(cat_cols)
        mat = np.zeros((n, n))
        for i, c1 in enumerate(cat_cols):
            for j, c2 in enumerate(cat_cols):
                if i == j:
                    mat[i, j] = 1.0
                else:
                    mat[i, j] = self._cramers_v(self.df[c1], self.df[c2])
        corr_df = pd.DataFrame(mat, index=cat_cols, columns=cat_cols)
        fig = px.imshow(corr_df, text_auto='.2f', aspect='auto',
                        color_continuous_scale='Viridis', zmin=0, zmax=1,
                        title="Cramér's V Categorical Association Heatmap",
                        template='plotly_white')
        fig.update_layout(height=600)
        fig.show()

    def plot_all_associations_heatmap(self):
        """Unified association heatmap across all column types.

        Uses:
          - Pearson r        for Num–Num pairs
          - Cramér's V       for Cat–Cat pairs
          - Point-Biserial / Eta (ANOVA) for mixed pairs
        """
        if self.df is None or self.df.empty:
            print('[plot_all_associations_heatmap] No data.'); return
        cols = self.df.columns.tolist()
        n = len(cols)
        mat = np.full((n, n), np.nan)
        for i, c1 in enumerate(cols):
            for j, c2 in enumerate(cols):
                mat[i, j] = self._association(c1, c2)
        assoc_df = pd.DataFrame(mat, index=cols, columns=cols)
        fig = px.imshow(assoc_df, text_auto='.2f', aspect='auto',
                        color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
                        title='Unified Association Heatmap (Pearson / Cramér\'s V / Eta)',
                        template='plotly_white')
        fig.update_layout(height=700)
        fig.show()

    # -----------------------------------------------------------------------
    # Internal statistical helpers
    # -----------------------------------------------------------------------
    @staticmethod
    def _cramers_v(x: pd.Series, y: pd.Series) -> float:
        """Compute Cramér's V between two categorical series."""
        contingency = pd.crosstab(x.fillna('_NA'), y.fillna('_NA'))
        chi2 = stats.chi2_contingency(contingency, correction=False)[0]
        n = contingency.values.sum()
        phi2 = chi2 / n
        r, k = contingency.shape
        phi2_corr = max(0, phi2 - (k-1)*(r-1)/(n-1))
        r_corr = r - (r-1)**2/(n-1)
        k_corr = k - (k-1)**2/(n-1)
        denom = min(k_corr-1, r_corr-1)
        return float(np.sqrt(phi2_corr / denom)) if denom > 0 else 0.0

    def _association(self, c1: str, c2: str) -> float:
        """Return appropriate association measure for a column pair."""
        is_num1 = pd.api.types.is_numeric_dtype(self.df[c1])
        is_num2 = pd.api.types.is_numeric_dtype(self.df[c2])
        s1, s2 = self.df[c1].dropna(), self.df[c2].dropna()
        idx = s1.index.intersection(s2.index)
        s1, s2 = s1.loc[idx], s2.loc[idx]
        if len(s1) < 2:
            return 0.0
        if c1 == c2:
            return 1.0
        if is_num1 and is_num2:
            return float(s1.corr(s2))
        if not is_num1 and not is_num2:
            return self._cramers_v(s1, s2)
        # Mixed – Point-Biserial if binary cat; Eta via ANOVA otherwise
        cat_s, num_s = (s1, s2) if not is_num1 else (s2, s1)
        groups = [num_s[cat_s == g].dropna().values for g in cat_s.unique()]
        groups = [g for g in groups if len(g) > 0]
        if len(groups) < 2:
            return 0.0
        if len(groups) == 2:
            r, _ = pointbiserialr(
                (cat_s == cat_s.unique()[0]).astype(int), num_s)
            return float(r)
        # Eta-squared
        grand_mean = num_s.mean()
        ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups)
        ss_total = ((num_s - grand_mean)**2).sum()
        eta2 = ss_between / ss_total if ss_total > 0 else 0.0
        return float(np.sqrt(eta2))  # return Eta (not Eta²) for [-1,1] scale

print('✅ DataInspector and PlottingMethods classes defined successfully.')

## 3. Full Demo – Titanic Dataset

### 3.1 Load Data

In [ ]:
inspector = DataInspector()

url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
inspector.df = pd.read_csv(url, na_values=_NULL_STRINGS, keep_default_na=True)
inspector._auto_convert_types()
print(f'Loaded Titanic dataset: {inspector.df.shape}')

### 3.2 Initial Inspection

In [ ]:
inspector.get_summary()

In [ ]:
inspector.column_details()

In [ ]:
inspector.show_missing_data()

### 3.3 Cleaning Pipeline

In [ ]:
# Impute numeric nulls with median
inspector.handle_missing_values(strategy='median')

# Impute categorical nulls with mode
inspector.handle_missing_values(strategy='mode')

# Remove duplicate rows
inspector.remove_duplicates()

In [ ]:
# Detect and remove outliers in numeric columns
inspector.handle_outliers(columns=['Age', 'Fare'], find_and_delete=True)

In [ ]:
# Drop columns with low predictive value for demo
inspector.delete_columns('Ticket,Cabin,Name')

### 3.4 Univariate Visualisations

In [ ]:
# 3-panel (Violin / Scatter / Histogram) for key numeric columns
inspector.plot_numerical(column_names=['Age', 'Fare', 'SibSp', 'Parch'])

In [ ]:
# Frequency bar charts for categorical columns
inspector.plot_categorical(column_names=['Sex', 'Embarked', 'Pclass'])

### 3.5 Relationship Plots (Auto-type Detection)

In [ ]:
# Num–Num → OLS Scatter
inspector.plot_relationship('Age', 'Fare')

In [ ]:
# Cat–Num → Box plot
inspector.plot_relationship('Sex', 'Fare')

In [ ]:
# Cat–Cat → Grouped bar
inspector.plot_relationship('Sex', 'Embarked')

### 3.6 Correlation & Association Heatmaps

In [ ]:
inspector.plot_numerical_correlation()

In [ ]:
inspector.plot_categorical_correlation()

In [ ]:
inspector.plot_all_associations_heatmap()

### 3.7 Feature Engineering – Normalize for ML

In [ ]:
scaled_num = inspector.extract_normalized_numeric_data(method='robust')
display(scaled_num.head())

In [ ]:
encoded_cat = inspector.extract_normalized_categorical_data(method='onehot')
display(encoded_cat.head())

In [ ]:
ml_ready = inspector.create_normalized_data_df(num_method='robust', cat_method='onehot')
display(ml_ready.head())
print('ML-ready DataFrame shape:', ml_ready.shape)

## 4. Custom Modular Plotting with `PlottingMethods`

In [ ]:
PLT = PlottingMethods()

# View all available plotting methods
pd.DataFrame(PLT.get_methods_info()['response'])

In [ ]:
# Pie chart – survival split
result = PLT.plot_pie_chart(names='Sex', values='Fare', hole=0.35,
                             title='Total Fare Paid by Gender',
                             data=inspector.df)
print('Status:', result.get('status'))
PLT.display_image(result)

In [ ]:
# Grouped bar chart – mean fare by Pclass and Sex
result = PLT.plot_bar_chart(x='Pclass', y='Fare', color='Sex',
                             barmode='group', title='Fare by Class & Gender',
                             data=inspector.df)
PLT.display_image(result)

In [ ]:
# Histogram – age distribution
result = PLT.plot_histogram(x='Age', bins=20,
                             title='Age Distribution', data=inspector.df)
PLT.display_image(result)

In [ ]:
# Heatmap – mean fare by Pclass × Embarked
result = PLT.plot_heat_map(values='Fare', index='Pclass', columns='Embarked',
                            aggregade_method='mean',
                            title='Mean Fare by Class × Embarkation',
                            data=inspector.df)
PLT.display_image(result)

In [ ]:
# Sankey diagram – embarkation → class flow
result = PLT.plot_sankey_diagram(source_column='Embarked',
                                  target_column='Sex',
                                  values='Fare',
                                  title='Fare Flow: Embarked → Sex',
                                  data=inspector.df)
print('Status:', result.get('status'))
PLT.display_image(result)

In [ ]:
# Sunburst – hierarchical view: Pclass → Sex
result = PLT.plot_simple_sunburst_graph(path=['Pclass', 'Sex'],
                                         values='Fare',
                                         title='Fare by Class & Gender (Sunburst)',
                                         data=inspector.df)
print('Status:', result.get('status'))
PLT.display_image(result)

---
## Summary

This notebook demonstrates a complete end-to-end data pipeline:

| Step | Method |
|------|--------|
| Load & sanitize | `upload_data()` / direct CSV read with `_NULL_STRINGS` |
| Inspect          | `get_summary()`, `column_details()`, `show_missing_data()` |
| Impute           | `handle_missing_values(strategy=…)` |
| Deduplicate      | `remove_duplicates()` |
| Outlier removal  | `handle_outliers(find_and_delete=True)` |
| Normalize        | `extract_normalized_numeric_data()` / `extract_normalized_categorical_data()` |
| ML-ready DF      | `create_normalized_data_df()` |
| Univariate viz   | `plot_numerical()`, `plot_categorical()` |
| Bivariate viz    | `plot_relationship()` (auto-selects chart type) |
| Correlations     | `plot_numerical_correlation()`, `plot_categorical_correlation()`, `plot_all_associations_heatmap()` |
| Custom charts    | `PlottingMethods` – bar, pie, histogram, heatmap, sankey, sunburst |
